# Brain Dance - Step 4: Integration Testing

This notebook tests the Mega-SAM → Instant4D pipeline on Google Colab.

**Prerequisites:**
- Google Colab Pro (T4/V100/A100 GPU)
- Your test video uploaded to Google Drive

**Estimated time:**
- First run: ~30 minutes (setup + compilation)
- Subsequent runs: ~15 minutes (pipeline only)

## Section A: Environment Setup

In [ ]:
# Cell 1: Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Clone repository with submodules
import os

if not os.path.exists('/content/brain-dance'):
    !git clone --recursive https://github.com/ujseah/brain-dance.git /content/brain-dance
else:
    print("Repository already cloned")
    
%cd /content/brain-dance

# Verify submodules are initialized
!ls -la instant4d/
!ls -la instant4d/SLAM/mega-sam/ 2>/dev/null || echo "Mega-SAM submodule not found - initializing..."
!git submodule update --init --recursive

In [ ]:
# Cell 3: Install base dependencies
print("Installing base dependencies...")
!pip install -q -r backend/requirements.txt

# Install GPU-specific packages
# Note: unidepth must be installed from GitHub, not PyPI
print("\nInstalling GPU packages...")
!pip install -q xformers plyfile gdown
!pip install -q git+https://github.com/lpiccinelli-eth/UniDepth.git

# Verify PyTorch CUDA
import torch
assert torch.cuda.is_available(), "CUDA not available after install!"
print(f"\nPyTorch CUDA: {torch.version.cuda}")

In [ ]:
# Cell 4: Compile Instant4D CUDA kernels (~10 minutes)
# This compiles: diff-gaussian-rasterization, pointops2, simple-knn, fused-ssim

print("Compiling Instant4D CUDA kernels...")
print("This takes ~10 minutes on first run.\n")

!bash scripts/setup_instant4d.sh

In [ ]:
# Cell 5: Setup Mega-SAM (~10 minutes)
# This compiles lietorch, droid_backends and downloads checkpoints

print("Setting up Mega-SAM...")
print("This downloads ~1.5GB of checkpoints and compiles CUDA extensions.\n")

!bash scripts/setup_megasam.sh

In [ ]:
# Cell 6: Verify all installations
print("Verifying installations...\n")

# Instant4D kernels
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("[OK] diff-gaussian-rasterization")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")

try:
    import pointops
    print("[OK] pointops")
except ImportError as e:
    print(f"[FAIL] pointops: {e}")

try:
    from simple_knn import distCUDA2
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")

try:
    from fused_ssim import fused_ssim
    print("[OK] fused-ssim")
except ImportError as e:
    print(f"[FAIL] fused-ssim: {e}")

# Mega-SAM kernels
try:
    import lietorch
    print("[OK] lietorch")
except ImportError as e:
    print(f"[FAIL] lietorch: {e}")

try:
    import droid_backends
    print("[OK] droid_backends")
except ImportError as e:
    print(f"[FAIL] droid_backends: {e}")

print("\nVerification complete!")

## Section B: Upload Your Video

In [ ]:
# Cell 7: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 8: Copy your video from Drive
import shutil
from pathlib import Path

# ============================================
# TODO: Update this path to your video file
# ============================================
VIDEO_PATH = "/content/drive/MyDrive/test_video.mp4"

# Copy to local storage for faster access
LOCAL_VIDEO = "/content/input.mp4"

if Path(VIDEO_PATH).exists():
    shutil.copy(VIDEO_PATH, LOCAL_VIDEO)
    print(f"Video copied to {LOCAL_VIDEO}")
else:
    print(f"ERROR: Video not found at {VIDEO_PATH}")
    print("Please update VIDEO_PATH to point to your video file.")

# Show video info
!ffprobe -v quiet -show_entries format=duration,size -show_entries stream=width,height,r_frame_rate -of csv=p=0 {LOCAL_VIDEO}

## Section C: Run Stage 1 (Mega-SAM Pose Estimation)

In [ ]:
# Cell 9: Process video with Mega-SAM
import sys
sys.path.insert(0, '/content/brain-dance/backend')

from stages.video_processing import VideoProcessingStage

# Configure Stage 1
config = {
    "pose_estimator": "megasam",  # Primary: Mega-SAM
    "max_frames": 150,             # Limit for testing (adjust based on video length)
    "megasam_opt_focal": True,     # Optimize focal length during SLAM
}

# Create stage and process video
stage1 = VideoProcessingStage(config)

print("Processing video with Mega-SAM...")
print("This includes: frame extraction, depth estimation, camera tracking\n")

stage1_result = stage1.process(
    video_path=LOCAL_VIDEO,
    output_dir="/content/stage1_output",
    progress_callback=lambda p, m: print(f"  [{p*100:.0f}%] {m}")
)

print(f"\n{'='*50}")
print(f"Stage 1 Complete!")
print(f"{'='*50}")
print(f"Frames extracted: {stage1_result.num_frames}")
print(f"Transforms: {stage1_result.transforms_path}")
print(f"Depth maps: {stage1_result.metadata.get('depth_maps_dir', 'N/A')}")
print(f"Motion prob: {stage1_result.metadata.get('motion_prob_path', 'N/A')}")

In [ ]:
# Cell 9b: Verify Stage 1 outputs
from pathlib import Path
import json

# Check transforms.json
transforms_path = Path(stage1_result.transforms_path)
if transforms_path.exists():
    with open(transforms_path) as f:
        transforms = json.load(f)
    print(f"Transforms: {len(transforms.get('frames', []))} frames")
    print(f"Camera model: {transforms.get('camera_model', 'N/A')}")
    if 'fl_x' in transforms:
        print(f"Focal length: {transforms['fl_x']:.1f}")

# Check depth maps (Mega-SAM specific)
depth_dir = stage1_result.metadata.get('depth_maps_dir')
if depth_dir and Path(depth_dir).exists():
    depth_files = list(Path(depth_dir).glob('*.npz'))
    print(f"\nDepth maps: {len(depth_files)} files")
else:
    print("\nNo Mega-SAM depth maps (using fallback path)")

# Check motion probability (Mega-SAM specific)
motion_path = stage1_result.metadata.get('motion_prob_path')
if motion_path and Path(motion_path).exists():
    import numpy as np
    motion_prob = np.load(motion_path)
    print(f"Motion probability: shape {motion_prob.shape}")
else:
    print("No Mega-SAM motion probability (using fallback)")

## Section D: Run Stage 3 (Instant4D 4D Training)

In [ ]:
# Cell 10: Train 4D Gaussians with Instant4D
from adapters.instant4d import Instant4DAdapter, Instant4DOptions

# Configure training options
options = Instant4DOptions(
    # Training parameters
    iterations=2000,        # Reduced for testing (default: 5000)
    batch_size=1,           # Images per batch
    
    # Mega-SAM integration
    use_megasam=True,       # Use depth/motion from Mega-SAM
    
    # Grid pruning
    enable_pruning=True,    # 92% Gaussian reduction
    
    # Export settings
    export_fps=10,          # 10 frames for quick test
)

# Create adapter and run pipeline
adapter = Instant4DAdapter()

print("Training 4D Gaussians with Instant4D...")
print(f"Iterations: {options.iterations}")
print(f"Export frames: {options.export_fps}\n")

stage3_result = adapter.run_full_pipeline(
    video_result=stage1_result,
    output_dir="/content/stage3_output",
    options=options,
    progress_callback=lambda p, m: print(f"  [{p*100:.0f}%] {m}")
)

print(f"\n{'='*50}")
print(f"Stage 3 Complete!")
print(f"{'='*50}")
print(f"Gaussians: {stage3_result.num_gaussians:,}")
print(f"PLY files: {len(stage3_result.ply_paths)}")
print(f"Model: {stage3_result.model_path}")
print(f"Metrics: {stage3_result.metrics}")

## Section E: Visualize Results

In [ ]:
# Cell 11: List output files
print("Stage 3 output files:")
!ls -la /content/stage3_output/

print("\nPer-frame PLY files:")
!ls -la /content/stage3_output/plys/ 2>/dev/null || echo "No plys/ directory"

In [ ]:
# Cell 12: Validate PLY files
from plyfile import PlyData
from pathlib import Path

ply_dir = Path("/content/stage3_output/plys")
if ply_dir.exists():
    ply_files = sorted(ply_dir.glob("*.ply"))
    print(f"Found {len(ply_files)} PLY files\n")
    
    # Check first and last frames
    for ply_path in [ply_files[0], ply_files[-1]]:
        ply = PlyData.read(str(ply_path))
        n_gaussians = len(ply['vertex'])
        print(f"{ply_path.name}: {n_gaussians:,} Gaussians")
        
    # Show Gaussian count distribution
    counts = []
    for ply_path in ply_files:
        ply = PlyData.read(str(ply_path))
        counts.append(len(ply['vertex']))
    
    print(f"\nGaussian count statistics:")
    print(f"  Min: {min(counts):,}")
    print(f"  Max: {max(counts):,}")
    print(f"  Mean: {sum(counts)/len(counts):,.0f}")
else:
    print("PLY directory not found!")

In [ ]:
# Cell 13: Check temporal metadata
if stage3_result.temporal_metadata:
    meta = stage3_result.temporal_metadata
    print("Temporal metadata:")
    print(f"  FPS: {meta.get('fps', 'N/A')}")
    print(f"  Duration: {meta.get('duration_seconds', 'N/A')} seconds")
    print(f"  Num frames: {meta.get('num_frames', 'N/A')}")
    
    timestamps = meta.get('timestamps', [])
    if timestamps:
        print(f"  Timestamps: {timestamps[:3]}...{timestamps[-3:]}")
else:
    print("No temporal metadata available")

## Section F: Save Results to Drive

In [ ]:
# Cell 14: Copy results to Google Drive
import shutil
from pathlib import Path
from datetime import datetime

# Create timestamped output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_output = Path(f"/content/drive/MyDrive/brain_dance_test/{timestamp}")
drive_output.mkdir(parents=True, exist_ok=True)

# Copy Stage 1 outputs
print("Copying Stage 1 outputs...")
shutil.copytree("/content/stage1_output", drive_output / "stage1", dirs_exist_ok=True)

# Copy Stage 3 outputs  
print("Copying Stage 3 outputs...")
shutil.copytree("/content/stage3_output", drive_output / "stage3", dirs_exist_ok=True)

print(f"\nResults saved to: {drive_output}")
!ls -la {drive_output}/

## Section G: Run Integration Tests (Optional)

In [ ]:
# Cell 15: Run pytest integration tests
%cd /content/brain-dance

print("Running setup verification tests...")
!pytest tests/test_instant4d_setup.py -v -m gpu --tb=short

print("\nRunning Instant4D adapter tests...")
!pytest tests/adapters/test_instant4d_adapter.py -v -m gpu --tb=short

## Summary

If all cells ran successfully, Step 4 integration testing is complete!

**What was tested:**
1. CUDA kernel compilation (Instant4D + Mega-SAM)
2. Stage 1: Mega-SAM pose estimation → depth maps + motion probability
3. Stage 3: Instant4D 4D training → per-frame PLY export

**Next steps:**
- Stage 5: Web export (PLY → SPZ compression)
- Frontend viewer with temporal playback